In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, event, text
from sqlalchemy.engine import URL

ROOT = r'D:\Bluestock work\bluestock-fintech-analysis1'
PROCESSED_DIR = os.path.join(ROOT, 'data', 'processed')
REPORTS_DIR = os.path.join(ROOT, 'reports')
DB_PATH = os.path.join(PROCESSED_DIR, 'bluestock_dw.db')

engine = create_engine(URL.create('sqlite', database=DB_PATH))

@event.listens_for(engine, 'connect')
def turn_on_foreign_keys(dbapi_connection, connection_record):
    cursor = dbapi_connection.cursor()
    cursor.execute('PRAGMA foreign_keys = ON')
    cursor.close()

In [3]:
returns = pd.read_sql_query("""
SELECT r.amfi_code, d.full_date, r.daily_return, r.days_since_prev
FROM fact_daily_return AS r
JOIN dim_date AS d ON d.date_key = r.date_key
ORDER BY r.amfi_code, r.date_key
""", engine)

returns['date'] = pd.to_datetime(returns['full_date'])

print(returns.shape)
print('Funds:', returns['amfi_code'].nunique())
print(returns['date'].min().date(), 'to', returns['date'].max().date())

(45960, 5)
Funds: 40
2022-01-04 to 2026-05-29


In [4]:
RF_ANNUAL = 0.065
TRADING_DAYS = 252
RF_DAILY = RF_ANNUAL / TRADING_DAYS

print('Daily risk-free rate (%):', round(RF_DAILY * 100, 5))

span = returns.groupby('amfi_code')['date'].agg(['min', 'max', 'count'])
span['years'] = (span['max'] - span['min']).dt.days / 365.25
span['obs_per_year'] = span['count'] / span['years']

print(span['obs_per_year'].describe().round(1))

Daily risk-free rate (%): 0.02579
count     40.0
mean     261.3
std        0.0
min      261.3
25%      261.3
50%      261.3
75%      261.3
max      261.3
Name: obs_per_year, dtype: float64


In [5]:
END_DATE = returns['date'].max()
first_date = returns.groupby('amfi_code')['date'].min()

COMMON_START = first_date.max()
years_common = (END_DATE - COMMON_START).days / 365.25

print('Common window:', COMMON_START.date(), 'to', END_DATE.date(), '| years:', round(years_common, 2))
print('Fund that starts last:', first_date.idxmax())

windows = {
    'common': returns[returns['date'] >= COMMON_START],
    'full': returns,
}

print(windows['common'].groupby('amfi_code').size().describe().round(0))

Common window: 2022-01-04 to 2026-05-29 | years: 4.4
Fund that starts last: 100016
count      40.0
mean     1149.0
std         0.0
min      1149.0
25%      1149.0
50%      1149.0
75%      1149.0
max      1149.0
dtype: float64


In [6]:
def build_blocks(window):
    return window.groupby('amfi_code')['daily_return'].agg(
        days='count',
        mean_daily='mean',
        std_daily='std',
        down_days=lambda s: (s < 0).sum(),
        down_std=lambda s: s[s < 0].std(),
        down_dev=lambda s: np.sqrt((np.minimum(s, 0) ** 2).mean()),
    )


blocks = {}
for name, window in windows.items():
    blocks[name] = build_blocks(window)

(blocks['common'] * [1, 100, 100, 1, 100, 100]).round(3).head(10)

,days,mean_daily,std_daily,down_days,down_std,down_dev
amfi_code,,,,,,
100016,1149,0.014,0.916,579,0.526,0.629
100025,1149,0.017,0.246,549,0.148,0.164
100033,1149,0.108,1.193,538,0.713,0.785
101206,1149,0.085,0.918,539,0.524,0.596
101207,1149,0.042,1.625,568,0.956,1.119
101208,1149,0.024,0.032,261,0.015,0.011
102885,1149,0.067,0.809,537,0.460,0.530
102886,1149,0.011,1.142,559,0.677,0.805
102887,1149,0.065,0.993,556,0.565,0.654
